In [ ]:
# Third-party imports
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from frozendict import frozendict

IS_FAST_MODE = os.environ.get("FLASQ_FAST_MODE_OVERRIDE", "False").lower() == "true"

# Local application imports
from qualtran.resource_counting import get_cost_value, QubitCount
from qualtran.surface_code.flasq.cirq_interop import convert_circuit_for_flasq_analysis
from qualtran.surface_code.flasq import cultivation_analysis
from qualtran.surface_code.flasq.error_mitigation import (  # type: ignore
    calculate_error_mitigation_metrics,
    calculate_failure_probabilities,
)
from qualtran.surface_code.flasq.examples.ising import build_ising_circuit
from qualtran.surface_code.flasq.flasq_model import (
    FLASQCostModel,
    apply_flasq_cost_model,
    conservative_FLASQ_costs,
    get_rotation_depth,
    optimistic_FLASQ_costs,
)
from qualtran.surface_code.flasq.measurement_depth import TotalMeasurementDepth
from qualtran.surface_code.flasq.examples import plotting as flasq_plt
from qualtran.surface_code.flasq.optimization import (
    CoreParametersConfig,
    ErrorBudget,
    generate_configs_for_constrained_qec,
    generate_configs_from_cultivation_data,
    post_process_for_failure_budget,
    post_process_for_pec_runtime,
    run_sweep,
)
from qualtran.surface_code.flasq.span_counting import TotalSpanCost
from qualtran.surface_code.flasq.symbols import (
    MIXED_FALLBACK_T_COUNT,
    ROTATION_ERROR,
    V_CULT_FACTOR,
    T_REACT,
)
from qualtran.surface_code.flasq.utils import substitute_until_fixed_point
from qualtran.surface_code.flasq.volume_counting import FLASQGateTotals

sns.set_context(context="paper", font_scale=1.1)

# FLASQ Resource Estimation for Ising Models

This notebook demonstrates the use of the FLASQ resource estimation framework for simulating Ising models. It includes single-point analyses, parameter sweeps, and comparisons against existing literature.

# Ising costs

In [ ]:
# --- Global Simulation Parameters ---
# These parameters are used for both the single-point analysis and the sweeps below.
if IS_FAST_MODE:
    rows = 5
    cols = 5
    n_steps = 5
else:
    rows = 11
    cols = 11
    n_steps = 20

total_allowable_rotation_error = 0.001
time_per_surface_code_cycle = 1e-6
reaction_time_in_cycles = 10.0
target_std_deviation = 0.0045

# --- Define circuit builder arguments (fixed for this script) ---
circuit_builder_kwargs_list = [
    frozendict(
        {
            "rows": rows,
            "cols": cols,
            "j_coupling": 1.0,
            "h_field": 3.04438,
            "dt": 0.04,
            "n_steps": n_steps,
            "periodic_boundary": True,
        }
    )
]

## Single-Point Analysis Function

This section defines a reusable function to perform a full FLASQ resource estimation for a single set of parameters. It encapsulates the logic for building the circuit, applying the cost model, resolving symbols, and printing a detailed summary.

In [ ]:
def run_and_print_single_point_analysis(
    name: str, circuit_kwargs: frozendict, analysis_params: dict
) -> dict:
    """Performs a full FLASQ resource estimation for a single set of parameters.

    Args:
        name: A descriptive name for this analysis point.
        circuit_kwargs: Keyword arguments to build the Ising circuit.
        analysis_params: Dictionary of analysis-specific parameters including:
            - physical_error_rate: Physical error rate of the qubits.
            - code_distance: The surface code distance.
            - target_cultivation_error_rate: The target error rate for T-gate cultivation.
            - n_total_logical_qubits: The total number of logical qubits available.
            - time_per_surface_code_cycle: Time duration of one surface code cycle in seconds.
            - reaction_time_in_cycles: Reaction time for magic state distillation in surface code cycles.
            - total_allowable_rotation_error: Total error budget for rotation synthesis.

    Returns:
        A dictionary containing key results from the analysis.
    """
    print(f"\n--- Single-Point Analysis: {name} ---")
    print("-" * (len(name) + 30))

    # Extract parameters
    physical_error_rate = analysis_params["physical_error_rate"]
    code_distance = analysis_params["code_distance"]
    target_cultivation_error_rate = analysis_params["target_cultivation_error_rate"]
    n_total_logical_qubits = analysis_params["n_total_logical_qubits"]
    time_per_surface_code_cycle = analysis_params["time_per_surface_code_cycle"]
    reaction_time_in_cycles = analysis_params["reaction_time_in_cycles"]
    total_allowable_rotation_error = analysis_params["total_allowable_rotation_error"]

    # 1. Find the best cultivation parameters from our pre-computed data.
    # We use target_cultivation_error_rate as the target_logical_error_rate for cultivation_analysis
    # to find the best cultivation strategy that yields this error rate.
    best_cult_params = cultivation_analysis.find_best_cultivation_parameters(
        physical_error_rate=physical_error_rate,
        target_logical_error_rate=target_cultivation_error_rate,
    )

    if best_cult_params.empty:
        raise ValueError(
            f"No suitable cultivation parameters found for the given inputs for {name}."
        )

    # 2. Derive cultivation factor and error rate from the data.
    cultivation_error_rate = best_cult_params["t_gate_cultivation_error_rate"]
    expected_volume = best_cult_params["expected_volume"]
    vcult_factor = expected_volume / (2 * (code_distance + 1) ** 2 * code_distance)

    # 3. Build and analyze the logical circuit.
    circuit = build_ising_circuit(**circuit_kwargs)
    cbloq, _ = convert_circuit_for_flasq_analysis(circuit)

    flasq_counts = get_cost_value(cbloq, FLASQGateTotals())
    total_span = get_cost_value(cbloq, TotalSpanCost())
    qubit_counts = get_cost_value(cbloq, QubitCount())

    ind_rot_err = total_allowable_rotation_error / flasq_counts.total_rotations
    rotation_depth_val = get_rotation_depth(rotation_error=ind_rot_err)
    measurement_depth = get_cost_value(
        cbloq, TotalMeasurementDepth(rotation_depth=rotation_depth_val)
    )

    # 4. Apply the FLASQ cost model.
    flasq_summary = apply_flasq_cost_model(
        model=conservative_FLASQ_costs,
        n_total_logical_qubits=n_total_logical_qubits,
        qubit_counts=qubit_counts,
        counts=flasq_counts,
        span_info=total_span,
        measurement_depth=measurement_depth,
        logical_timesteps_per_measurement=reaction_time_in_cycles / code_distance,
    )

    # 5. Resolve symbols and calculate final metrics.
    resolved_flasq_summary = flasq_summary.resolve_symbols(
        frozendict(
            {
                ROTATION_ERROR: ind_rot_err,
                V_CULT_FACTOR: vcult_factor,

                T_REACT: reaction_time_in_cycles / code_distance,
            }
        )
    )

    eff_time, wall_time, _ = calculate_error_mitigation_metrics(
        flasq_summary=resolved_flasq_summary,
        time_per_surface_code_cycle=time_per_surface_code_cycle,
        code_distance=code_distance,
        lambda_val=0.01 / physical_error_rate,
        cultivation_error_rate=cultivation_error_rate,
    )

    # Calculate logical error rate (P_fail_Clifford + P_fail_T)
    P_fail_Clifford, P_fail_T = calculate_failure_probabilities(
        flasq_summary=resolved_flasq_summary,
        code_distance=code_distance,
        lambda_val=0.01 / physical_error_rate,
        cultivation_error_rate=cultivation_error_rate,
    )
    logical_error_rate = 1 - (1 - P_fail_Clifford) * (1 - P_fail_T)

    # Calculate non-Clifford cost contributions for detailed reporting
    total_non_clifford_volume = (
        resolved_flasq_summary.non_clifford_lattice_surgery_volume
        + resolved_flasq_summary.cultivation_volume
    )
    frac_comp_vol_non_clifford = (
        total_non_clifford_volume / resolved_flasq_summary.total_computational_volume
    )
    frac_spacetime_vol_non_clifford = (
        total_non_clifford_volume / resolved_flasq_summary.total_spacetime_volume
    )
    frac_non_clifford_vol_cultivation = (
        resolved_flasq_summary.cultivation_volume / total_non_clifford_volume
    )

    # 6. Print a clean summary of the results.
    n_phys_qubits_total = n_total_logical_qubits * 2 * (code_distance + 1) ** 2
    print(f"  Circuit: {circuit_kwargs}")
    print(f"  Input Physical Error Rate: {physical_error_rate:.2e}")
    print(f"  Input Code Distance: {code_distance}")
    print(f"  Target Cultivation Error Rate: {target_cultivation_error_rate:.2e}")
    print("-" * 35)
    print("  Derived Cultivation Parameters:")
    print(
        f"    Source Cultivation Distance: {best_cult_params['cultivation_distance']}"
    )
    print(f"    Derived V_CULT Factor: {vcult_factor:.2f}")
    print(f"    Resulting Cultivation Error Rate: {cultivation_error_rate:.2e}")
    print("-" * 35)
    print("  Resource Summary:")
    print(f"    Algorithmic Qubits: {resolved_flasq_summary.n_algorithmic_qubits}")
    print(f"    Fluid Ancilla Qubits: {resolved_flasq_summary.n_fluid_ancilla}")
    print(f"    Total Logical Qubits: {n_total_logical_qubits}")
    print(f"    Total Physical Qubits Required: {n_phys_qubits_total:,.0f}")
    print(f"    Total T-gates: {resolved_flasq_summary.total_t_count:,.0f}")
    print(f"    Total Rotations: {resolved_flasq_summary.total_rotation_count:,.0f}")
    print(f"    Total Logical Depth: {resolved_flasq_summary.total_depth:,.2f}")
    print(
        f"    Total Spacetime Volume: {resolved_flasq_summary.total_spacetime_volume:,.2f}"
    )
    print("-" * 35)
    print("  Volume Breakdown:")
    print(
        f"    Total Computational Volume: {resolved_flasq_summary.total_computational_volume:,.2f}"
    )
    print(
        f"    Total Spacetime Volume: {resolved_flasq_summary.total_spacetime_volume:,.2f}"
    )
    print(f"    Total Non-Clifford Volume: {total_non_clifford_volume:,.2f}")
    print(f"      Fraction of Computational Volume: {frac_comp_vol_non_clifford:.2%}")
    print(
        f"      Fraction of Non-Clifford from Cultivation: {frac_non_clifford_vol_cultivation:.2%}"
    )
    print(f"      Fraction of Spacetime Volume: {frac_spacetime_vol_non_clifford:.2%}")
    print("-" * 35)
    print("  Final Performance Metrics:")
    print(f"    Effective Time per Sample: {eff_time:.2f} s")
    print(f"    Wall Clock Time per Sample: {wall_time:.2f} s")
    print(f"    Overall Logical Error Rate: {logical_error_rate:.2e}")
    print("-" * 35)

    return {
        "name": name,
        "physical_qubits": n_phys_qubits_total,
        "flasq_time_per_sample": eff_time,
        "wall_time_per_sample": wall_time,
        "logical_qubits": n_total_logical_qubits,
        "code_distance": code_distance,
        "logical_error_rate": logical_error_rate,
        "cultivation_error_rate": cultivation_error_rate,
        "total_depth": resolved_flasq_summary.total_depth,
        "total_spacetime_volume": resolved_flasq_summary.total_spacetime_volume,
    }

## Single-Point Examples for LaTeX Table

We now use the `run_and_print_single_point_analysis` function to generate the data for the LaTeX table.

In [ ]:
# --- Example 1: 11x11 Ising Model (2nd Order Trotter) ---
circuit_kwargs_11x11 = frozendict(
    {
        "rows": 11,
        "cols": 11,
        "j_coupling": 1.0,
        "h_field": 3.04438,
        "dt": 0.04,
        "n_steps": n_steps,  # From global parameters (20 steps)
        "order": 2,  # Default for build_ising_circuit
    }
)

analysis_params_11x11 = {
    "physical_error_rate": 1e-3,
    "code_distance": 13,
    "target_cultivation_error_rate": 3e-7,
    "n_total_logical_qubits": 160,
    "time_per_surface_code_cycle": time_per_surface_code_cycle,  # From global parameters
    "reaction_time_in_cycles": reaction_time_in_cycles,  # From global parameters
    "total_allowable_rotation_error": total_allowable_rotation_error,  # From global parameters
}

results_11x11 = run_and_print_single_point_analysis(
    "11x11 Ising Model (2nd Order Trotter)",
    circuit_kwargs_11x11,
    analysis_params_11x11,
)

In [ ]:
# --- Example 2: 10x10 Ising Model (4th Order Trotter) for Beverland Comparison ---
# This uses parameters from the Beverland comparison section later in the notebook.
circuit_kwargs_10x10_bev = frozendict(
    {
        "rows": 5 if IS_FAST_MODE else 10,
        "cols": 5 if IS_FAST_MODE else 10,
        "j_coupling": 1.0,
        "h_field": 0.5,
        "dt": 0.1,
        "n_steps": 5 if IS_FAST_MODE else 20,
        "order": 4,
    }
)

analysis_params_10x10_bev = {
    "physical_error_rate": 1e-3,
    "code_distance": 14,
    "target_cultivation_error_rate": 2e-7,
    "n_total_logical_qubits": 140,
    "time_per_surface_code_cycle": 400e-9,  # From Beverland section
    "reaction_time_in_cycles": reaction_time_in_cycles,  # From Beverland section
    "total_allowable_rotation_error": 0.001,  # From Beverland section
}

results_10x10_bev = run_and_print_single_point_analysis(
    "10x10 Ising Model (4th Order Trotter) - Beverland Comparison",
    circuit_kwargs_10x10_bev,
    analysis_params_10x10_bev,
)

## LaTeX Table Data Summary

Here are the values to fill out your LaTeX table:

In [ ]:
print("\n--- Data for LaTeX Table ---")
print("10x10 Ising Model (4th Order Trotter):")
print(f"  Physical qubits: {results_10x10_bev['physical_qubits']:,}")
print(
    f"  FLASQ time per sample (wall time): {results_10x10_bev['wall_time_per_sample']:.2f} seconds"
)
print(
    f"  FLASQ time per effective sample (PEC): {results_10x10_bev['flasq_time_per_sample']:.2f} seconds"
)
print(f"  Logical qubits: {results_10x10_bev['logical_qubits']}")
print(f"  Logical depth: {results_10x10_bev['total_depth']:,.2f}")
print(f"  Code distance: {results_10x10_bev['code_distance']}")
print(f"  Overall logical error rate: {results_10x10_bev['logical_error_rate']:.2e}")
print(f"  Cultivation error rate: {results_10x10_bev['cultivation_error_rate']:.2e}")
print(f"  Total spacetime volume: {results_10x10_bev['total_spacetime_volume']:,.2f}")
print("\n11x11 Ising Model (2nd Order Trotter):")
print(f"  Physical qubits: {results_11x11['physical_qubits']:,}")
print(
    f"  FLASQ time per sample (wall time): {results_11x11['wall_time_per_sample']:.2f} seconds"
)
print(
    f"  FLASQ time per effective sample (PEC): {results_11x11['flasq_time_per_sample']:.2f} seconds"
)
print(f"  Logical qubits: {results_11x11['logical_qubits']}")
print(f"  Logical depth: {results_11x11['total_depth']:,.2f}")
print(f"  Code distance: {results_11x11['code_distance']}")
print(f"  Overall logical error rate: {results_11x11['logical_error_rate']:.2e}")
print(f"  Cultivation error rate: {results_11x11['cultivation_error_rate']:.2e}")
print(f"  Total spacetime volume: {results_11x11['total_spacetime_volume']:,.2f}")
print("-" * 30)

# Run the calculations for a plot

In [ ]:

# --- Define parameter ranges for the sweep ---
if IS_FAST_MODE:
    phys_error_rate_list_sweep = [0.001, 0.002]
    code_distance_list_sweep = [13, 15]
else:
    phys_error_rate_list_sweep = np.logspace(
        np.log2(0.00025), np.log2(0.002), base=2, num=13
    )
    code_distance_list_sweep = list(range(4, 31, 1))

# --- Generate CoreParametersConfig list using cultivation data ---
# This will derive cultivation_error_rate and vcult_factor based on
# the cultivation data for each (code_distance, phys_error_rate) pair.

if IS_FAST_MODE:
    core_configs_list = [
        CoreParametersConfig(
            code_distance=5,
            phys_error_rate=1e-4,
            cultivation_error_rate=1e-6,
            vcult_factor=3.0,
            cultivation_data_source_distance=5,
            target_t_error=1e-6,
        ),
        CoreParametersConfig(
            code_distance=15,
            phys_error_rate=1e-2,
            cultivation_error_rate=1e-4,
            vcult_factor=3.5,
            cultivation_data_source_distance=5,
            target_t_error=1e-4,
        ),
    ]
else:
    core_configs_list = generate_configs_from_cultivation_data(
        code_distance_list=code_distance_list_sweep,
        phys_error_rate_list=phys_error_rate_list_sweep,
        cultivation_data_source_distance_list=[3, 5],  # Use data from both dist 3 and 5
        cultivation_data_sampling_frequency=5,
    )

# --- Other sweep parameters ---
if IS_FAST_MODE:
    n_phys_qubits_total_list_sweep = [12500, 400000]
else:
    n_phys_qubits_total_list_sweep = [
        int(np.floor(n + 0.5))
        for n in np.logspace(
            np.log10(12500), np.log10(400000), num=16  #
        )  # Original denser sweep
    ]
flasq_model_configs_list_sweep = [
    (conservative_FLASQ_costs, "Conservative"),
    (optimistic_FLASQ_costs, "Optimistic"),
]

sweep_results = run_sweep(
    circuit_builder_func=build_ising_circuit,
    circuit_builder_kwargs_list=circuit_builder_kwargs_list,
    core_configs_list=core_configs_list,
    total_allowable_rotation_error_list=[total_allowable_rotation_error],
    n_phys_qubits_total_list=n_phys_qubits_total_list_sweep,
    flasq_model_configs=flasq_model_configs_list_sweep,
    reaction_time_in_cycles_list=[reaction_time_in_cycles],
    print_level=1,  # Show progress bar
)

In [ ]:
ising_scan_df = post_process_for_pec_runtime(
    sweep_results, time_per_surface_code_cycle=time_per_surface_code_cycle
)

### Main Heatmap Analysis
This section takes the raw data from the parameter sweep and uses the generic plotting functions to generate two heatmaps:
1.  **Time to Solution**: The estimated time in hours to reach the target precision.
2.  **Optimal Code Distance**: The code distance `d` that yields the best (regularized) time to solution.

In [ ]:

# 1. Enrich the DataFrame with calculated metrics for plotting.
df_enriched = flasq_plt.enrich_sweep_df(
    ising_scan_df, target_std_dev=target_std_deviation
)

# 2. Filter the data for the 'Conservative' model and specific axis ranges.
df_conservative = df_enriched[df_enriched["FLASQ Model"] == "Conservative"]
df_filtered = df_conservative[
    (df_conservative["Effective Time per Sample (s)"] > 0)
    & (df_conservative["Time to Solution (hr)"] <= 1000)
    & (df_conservative["Physical Error Rate"] >= 0.01 / 21)
    & (df_conservative["Number of Physical Qubits"] <= 400000)
]

# 3. Find the optimal configuration (code distance) for the Time to Solution plot.
#    For this plot, we want the absolute minimum time, so no regularization is used.
df_optimal_time = flasq_plt.find_optimal_heatmap_configs(
    df_filtered,
    x_axis_col="Number of Physical Qubits",
    y_axis_col="Physical Error Rate",
    # y_axis_col="Lambda",
    value_col_to_optimize="Time to Solution (hr)",
    regularization_col=None,  # No regularization for the time plot
)

# 4. Plot the "Time to Solution" heatmap.
if not df_optimal_time.empty:
    fig, ax = flasq_plt.plot_flasq_heatmap(
        df_optimal_time,
        x_axis_col="Number of Physical Qubits",
        y_axis_col="Physical Error Rate",
        # y_axis_col="Lambda",
        value_col_to_plot="Time to Solution (hr)",
        title="Time to Estimate $\\langle Z_{tot}^2 \\rangle$ in an $11 \\times 11$ Ising Model",
        invert_yaxis=True,
        vmin=1,
        vmax=1000,
        cbar_label="Time to solution (hrs)",
    )

    ax.set_ylabel("Physical error rate")
    ax.set_xlabel("Number of physical qubits")
else:
    print("Skipping Time to Solution heatmap because data is empty.")

plt.savefig("conservative_ising_heatmap.pdf", bbox_inches="tight")
plt.tight_layout()
plt.show()

In [ ]:
# 3. Find the optimal configuration for the Code Distance plot.
#    Here, we use regularization to prefer smaller code distances when
#    runtimes are very similar.
df_optimal_dist = flasq_plt.find_optimal_heatmap_configs(
    df_filtered,
    x_axis_col="Number of Physical Qubits",
    y_axis_col="Physical Error Rate",
    # y_axis_col="Lambda",
    value_col_to_optimize="Time to Solution (hr)",
    # regularization_col="Code Distance",  # Use default regularization
)

# 4. Plot the "Optimal Code Distance" heatmap.
if not df_optimal_dist.empty:
    fix, ax = flasq_plt.plot_flasq_heatmap(
        df_optimal_dist,
        x_axis_col="Number of Physical Qubits",
        y_axis_col="Physical Error Rate",
        # y_axis_col="Lambda",
        value_col_to_plot="Code Distance",
        title="Optimal Code Distance for Ising Simulations (Conservative FLASQ Estimates)",
        invert_yaxis=True,
        log_scale=False,
        cmap="flare_r",
        cbar_label="Code distance",
        skip_decimal_formatting=True,
    )

    ax.set_ylabel("Physical error rate")
    ax.set_xlabel("Number of physical qubits")
else:
    print("Skipping Optimal Code Distance heatmap because data is empty.")

plt.savefig("conservative_code_distance_ising_heatmap.pdf", bbox_inches="tight")
plt.tight_layout()
plt.show()

### FLASQ Estimates with Open Boundary Conditions

We now repeat the analysis but for a system with open boundary conditions (OBC). This is more representative of what might be implemented on a near-term device without costly long-range interactions. The resulting heatmap will serve as the basis for a more direct comparison with the NISQ cost model.

In [ ]:
# --- Generate Data for Open Boundary Conditions ---

# 1. Define circuit builder arguments for OBC.
circuit_builder_kwargs_list_obc = [
    frozendict(d, periodic_boundary=False) for d in circuit_builder_kwargs_list
]

# 2. Run the sweep with the new OBC arguments.
sweep_results_obc = run_sweep(
    circuit_builder_func=build_ising_circuit,
    circuit_builder_kwargs_list=circuit_builder_kwargs_list_obc,
    core_configs_list=core_configs_list,
    total_allowable_rotation_error_list=[total_allowable_rotation_error],
    n_phys_qubits_total_list=n_phys_qubits_total_list_sweep,
    flasq_model_configs=flasq_model_configs_list_sweep,
    reaction_time_in_cycles_list=[reaction_time_in_cycles],
    print_level=1,
)


# --- Generate Data for Open Boundary Conditions ---

# 1. Define circuit builder arguments for OBC.
circuit_builder_kwargs_list_obc_deep = [
    frozendict(d, periodic_boundary=False, n_steps=40)
    for d in circuit_builder_kwargs_list_obc
]

# 2. Run the sweep with the new OBC arguments.
sweep_results_obc_deep = run_sweep(
    circuit_builder_func=build_ising_circuit,
    circuit_builder_kwargs_list=circuit_builder_kwargs_list_obc_deep,
    core_configs_list=core_configs_list,
    total_allowable_rotation_error_list=[total_allowable_rotation_error],
    n_phys_qubits_total_list=n_phys_qubits_total_list_sweep,
    flasq_model_configs=flasq_model_configs_list_sweep,
    reaction_time_in_cycles_list=[reaction_time_in_cycles],
    print_level=1,
)


# 3. Post-process the OBC results.
ising_scan_df_obc = post_process_for_pec_runtime(
    sweep_results_obc, time_per_surface_code_cycle=time_per_surface_code_cycle
)
df_enriched_obc = flasq_plt.enrich_sweep_df(
    ising_scan_df_obc, target_std_dev=target_std_deviation
)

In [ ]:
# --- Plot the OBC Time to Solution Heatmap ---

# 1. Filter the enriched OBC data for the 'Conservative' model.
df_conservative_obc = df_enriched_obc[df_enriched_obc["FLASQ Model"] == "Conservative"]
df_filtered_obc = df_conservative_obc[
    (df_conservative_obc["Effective Time per Sample (s)"] > 0)
    & (df_conservative_obc["Time to Solution (hr)"] <= 1000)
    & (df_conservative_obc["Physical Error Rate"] >= 0.01 / 31)
    & (df_conservative_obc["Number of Physical Qubits"] <= 400000)
]

# 2. Find the optimal configuration for the OBC time plot.
df_optimal_time_obc = flasq_plt.find_optimal_heatmap_configs(
    df_filtered_obc,
    x_axis_col="Number of Physical Qubits",
    y_axis_col="Physical Error Rate",
    value_col_to_optimize="Time to Solution (hr)",
    regularization_col=None,
)

# 3. Plot the heatmap for OBC.
fig, ax = flasq_plt.plot_flasq_heatmap(
    df_optimal_time_obc,
    x_axis_col="Number of Physical Qubits",
    y_axis_col="Physical Error Rate",
    value_col_to_plot="Time to Solution (hr)",
    title="Time to Estimate $\\langle Z_{tot}^2 \\rangle$ (Open Boundary Conditions)",
    invert_yaxis=True,
    vmin=1,
    vmax=1000,
    cbar_label="Time to solution (hrs)",
)
ax.set_ylabel("Physical error rate")
ax.set_xlabel("Number of physical qubits")
plt.savefig("conservative_ising_heatmap_obc.pdf", bbox_inches="tight")
plt.tight_layout()
plt.show()

In [ ]:

# 2. Filter the data for the 'Optimistic' model and specific axis ranges.
df_optimistic = df_enriched[df_enriched["FLASQ Model"] == "Optimistic"]
df_filtered = df_optimistic[
    (df_optimistic["Effective Time per Sample (s)"] > 0)
    & (df_optimistic["Time to Solution (hr)"] <= 1000)
    & (df_optimistic["Physical Error Rate"] >= 4e-4)
    & (df_optimistic["Number of Physical Qubits"] <= 400000)
]

# 3. Find the optimal configuration (code distance) for the Time to Solution plot.
#    For this plot, we want the absolute minimum time, so no regularization is used.
df_optimal_time = flasq_plt.find_optimal_heatmap_configs(
    df_filtered,
    x_axis_col="Number of Physical Qubits",
    y_axis_col="Physical Error Rate",
    # y_axis_col="Lambda",
    value_col_to_optimize="Time to Solution (hr)",
    regularization_col=None,  # No regularization for the time plot
)

# 4. Plot the "Time to Solution" heatmap.
fig, ax = flasq_plt.plot_flasq_heatmap(
    df_optimal_time,
    x_axis_col="Number of Physical Qubits",
    y_axis_col="Physical Error Rate",
    # y_axis_col="Lambda",
    value_col_to_plot="Time to Solution (hr)",
    title="Time to Estimate $\\langle Z_{tot}^2 \\rangle$ in an $11 \\times 11$ Ising Model (Optimistic FLASQ Estimates)",
    invert_yaxis=True,
    vmin=1,
    vmax=1000,
    cbar_label="Time to solution (hrs)",
)

ax.set_ylabel("Physical error rate")
ax.set_xlabel("Number of physical qubits")

plt.savefig("optimistic_ising_heatmap.pdf", bbox_inches="tight")
plt.tight_layout()
plt.show()

# What about just 10k qubits?

In [ ]:
# --- Define circuit builder arguments for different side_lengths ---
if IS_FAST_MODE:
    side_lengths = [4, 5]
else:
    side_lengths = range(4, 11)

circuit_builder_kwargs_list = [
    frozendict(
        {
            "rows": side_length,
            "cols": side_length,
            "j_coupling": 1.0,
            "h_field": 3.04438,
            "dt": 0.04,
            "n_steps": n_steps,
        }
    )
    for side_length in side_lengths
]

# --- Define parameter ranges for the sweep ---
if IS_FAST_MODE:
    phys_error_rate_list = [1e-4, 1e-2]
    code_distance_list = [5, 15]
else:
    phys_error_rate_list = np.logspace(-4, -2, num=31)
    code_distance_list = list(range(5, 40, 2))

# --- Generate CoreParametersConfig list using cultivation data ---
# This will derive cultivation_error_rate and vcult_factor based on
# the cultivation data for each (code_distance, phys_error_rate) pair.
# You can adjust cultivation_data_source_distance_list and
# cultivation_data_sampling_frequency as needed.
if IS_FAST_MODE:
    core_configs_list = [
        CoreParametersConfig(
            code_distance=5,
            phys_error_rate=1e-4,
            cultivation_error_rate=1e-6,
            vcult_factor=3.0,
            cultivation_data_source_distance=5,
            target_t_error=1e-6,
        ),
        CoreParametersConfig(
            code_distance=15,
            phys_error_rate=1e-2,
            cultivation_error_rate=1e-4,
            vcult_factor=3.5,
            cultivation_data_source_distance=5,
            target_t_error=1e-4,
        ),
    ]
else:
    core_configs_list = generate_configs_from_cultivation_data(
        code_distance_list=code_distance_list,
        phys_error_rate_list=phys_error_rate_list,
        cultivation_data_source_distance_list=[
            3,
            5,
        ],  # Example: use data from both dist 3 and 5
        cultivation_data_sampling_frequency=5,  # Example: sample every 10th data point from tail
    )


# --- Other sweep parameters (mostly unchanged) ---
flasq_model_configs_list = [
    (conservative_FLASQ_costs, "Conservative"),
    (optimistic_FLASQ_costs, "Optimistic"),
]
# Fixed total physical qubits for this sweep
n_phys_qubits_total_list = [10000]


# --- Run the optimization sweep ---
sweep_results = run_sweep(
    circuit_builder_func=build_ising_circuit,
    circuit_builder_kwargs_list=circuit_builder_kwargs_list,
    core_configs_list=core_configs_list,
    total_allowable_rotation_error_list=[total_allowable_rotation_error],
    n_phys_qubits_total_list=n_phys_qubits_total_list,
    flasq_model_configs=flasq_model_configs_list,
    reaction_time_in_cycles_list=[reaction_time_in_cycles],
    print_level=1,  # Optional: to show a progress bar
)

ising_scan_df = post_process_for_pec_runtime(
    sweep_results, time_per_surface_code_cycle=time_per_surface_code_cycle
)
ising_scan_df = flasq_plt.enrich_sweep_df(ising_scan_df, target_std_dev=0.005)

In [ ]:
filtered_df = ising_scan_df[
    (ising_scan_df["Effective Time per Sample (s)"] > 0)
    & (ising_scan_df["Time to Solution (hr)"] <= 1000)
]
# filtered_df = filtered_df[filtered_df["flasq_model"] == "Conservative"] # Optional filter
min_time_indices = filtered_df.groupby(
    ["circuit_arg_rows", "Physical Error Rate", "FLASQ Model"]
)["Time to Solution (hr)"].idxmin()

# Use the collected indices to select only the rows with the optimal times.
min_time_df = filtered_df.loc[min_time_indices]


# --- Plotting Code (using the new 'min_time_df') ---

sns.set_theme(style="whitegrid")
g = sns.relplot(
    data=min_time_df,  # Use the filtered DataFrame here
    kind="line",
    # kind="scatter",
    x="Physical Error Rate",
    y="Time to Solution (hr)",
    hue="circuit_arg_rows",
    col="FLASQ Model",
    palette="tab10",
    height=6,
    aspect=0.8,
)

g.set(xscale="log", yscale="log")
g.set_titles("Cost Assumptions = {col_name}")
g.fig.suptitle(
    "Time to Solution for R x R Ising Models (Optimal Code Distance)",
    y=1.03,
    # fontsize=16,
)
g.legend.set_title("Side Length")

# --- Add and Customize Minor Ticks ---
# Iterate through each subplot (Axes object) in the FacetGrid
for ax in g.axes.flat:
    ax.grid(True, which="minor", axis="both", linestyle=":", linewidth=0.7, alpha=0.6)

plt.tight_layout()
plt.show()

## NISQ Cost Comparison

In this section, we add an estimate for the runtime of the same simulation on a NISQ-era device using Probabilistic Error Cancellation (PEC) as a baseline for comparison against the fault-tolerant estimates.

Our model for the NISQ cost makes the following assumptions:
- **Circuit Structure**: The 2nd-order Trotterized evolution is decomposed into layers of single-qubit X rotations (depth 1) and two-qubit ZZ interactions (depth 4). Merging steps between Trotter blocks gives a total circuit depth of `5 * n_steps + 1`.
- **Noise Model**: Each physical gate is modeled as an ideal gate followed by a single-qubit depolarizing channel with strength `epsilon = p_phys`.
- **PEC Overhead**: The sampling overhead is calculated using the optimal formula for PEC under depolarizing noise. The total overhead `gamma_tot` is the per-gate overhead raised to the power of the total number of gate opportunities (qubits * depth).
- **Sampling Cost**: The number of shots `N` required to reach a `target_std_dev` scales as `gamma_tot**2 / target_std_dev**2`.
- **Parallelization**: We assume we can run multiple instances of the experiment in parallel, limited by the number of physical qubits available on the device. The total runtime is divided by this parallelization factor.

In [ ]:
import math


def get_NISQ_cost(
    n_phys_qubits: int,
    p_phys: float,
    system_size: int,
    n_trotter_steps: int,
    target_std_dev: float,
    two_qubit_errors_only: bool = False,
) -> float:
    """Estimates the runtime for a NISQ device using Probabilistic Error Cancellation (PEC).

    Args:
        n_phys_qubits: The total number of physical qubits available on the device.
        p_phys: The physical error rate, used as the depolarizing strength `epsilon`.
        system_size: The number of qubits required for one instance of the simulation.
        n_trotter_steps: The number of Trotter steps in the simulation.
        target_std_dev: The target standard deviation for the final expectation value.

    Returns:
        The estimated total runtime in hours. Returns float('inf') if parameters are invalid.
    """
    if p_phys >= 1.0:
        return float("inf")

    # 1. Model the circuit execution
    if two_qubit_errors_only:
        total_depth = 4 * n_trotter_steps
    else:
        total_depth = 5 * n_trotter_steps + 1

    total_gate_opportunities = system_size * total_depth

    # 2. Calculate PEC Overhead
    epsilon = p_phys
    gamma_per_gate = (1 + epsilon / 2) / (1 - epsilon)
    gamma_tot = gamma_per_gate**total_gate_opportunities

    # 3. Calculate Total Runtime
    n_samples = gamma_tot**2 / target_std_dev**2
    time_per_step_s = 50e-9  # Assume same as surface code cycle for consistency
    runtime_single_instance_s = n_samples * total_depth * time_per_step_s

    # 4. Account for Parallelization
    n_parallel = math.floor(n_phys_qubits / system_size)
    if n_parallel == 0:
        return float("inf")

    total_runtime_s = runtime_single_instance_s / n_parallel
    return total_runtime_s / 3600  # Convert to hours


def add_nisq_costs_to_dataframe(
    df: pd.DataFrame,
    system_size: int,
    n_trotter_steps: int,
    target_std_dev: float,
    two_qubit_errors_only: bool = False,
) -> pd.DataFrame:
    """Adds a 'NISQ Time to Solution (hr)' column to the DataFrame."""
    if df.empty:
        df["NISQ Time to Solution (hr)"] = pd.Series(dtype=float)
        return df
        
    df["NISQ Time to Solution (hr)"] = df.apply(
        lambda row: get_NISQ_cost(
            row["Number of Physical Qubits"],
            row["Physical Error Rate"],
            system_size,
            n_trotter_steps,
            target_std_dev,
            two_qubit_errors_only,
        ),
        axis=1,
    )
    return df

## Fault-Tolerant vs. NISQ Runtime Comparison

In this section, we visualize the NISQ runtime and then compare it directly to the optimal fault-tolerant runtime via a ratio plot. This helps to identify the parameter regimes where each paradigm is more efficient.

In [ ]:
from matplotlib.colors import LinearSegmentedColormap
import matplotlib.cm as cm


# --- Augment Optimal DataFrame with NISQ Costs ---
# This cell now uses the OBC data (`df_conservative_obc`) as the baseline for comparison.
df_optimal_time_obc_filtered = df_conservative_obc[
    (df_conservative_obc["Effective Time per Sample (s)"] > 0)
    & (df_conservative_obc["Time to Solution (hr)"] > 0)
    & (df_conservative_obc["Time to Solution (hr)"] <= 1e30)
    & (df_conservative_obc["Physical Error Rate"] >= 4e-4)
    & (df_conservative_obc["Number of Physical Qubits"] <= 400000)
]

# Find the optimal FT configurations from the OBC data.
df_optimal_ft_obc = flasq_plt.find_optimal_heatmap_configs(
    df_optimal_time_obc_filtered,
    x_axis_col="Number of Physical Qubits",
    y_axis_col="Physical Error Rate",
    value_col_to_optimize="Time to Solution (hr)",
    regularization_col=None,
)


df_optimal_time_with_nisq = add_nisq_costs_to_dataframe(
    df_optimal_ft_obc.copy(),
    system_size=rows * cols,
    n_trotter_steps=n_steps,
    target_std_dev=target_std_deviation,
    two_qubit_errors_only=True,
)

df_optimal_time_with_nisq = df_optimal_time_with_nisq[
    (df_optimal_time_with_nisq["NISQ Time to Solution (hr)"] > 0)
    & (df_optimal_time_with_nisq["NISQ Time to Solution (hr)"] < 1e30)
    & (
        (df_optimal_time_with_nisq["Time to Solution (hr)"] < 1000)
        | (df_optimal_time_with_nisq["NISQ Time to Solution (hr)"] < 1000)
    )
    & (df_optimal_time_with_nisq["Number of Physical Qubits"] >= 22000)
]

df_optimal_time_with_nisq_only = df_optimal_time_with_nisq[
    (df_optimal_time_with_nisq["NISQ Time to Solution (hr)"] < 1e3)
]

# --- Plot NISQ-only Heatmap ---
fig, ax = flasq_plt.plot_flasq_heatmap(
    df_optimal_time_with_nisq_only,
    x_axis_col="Number of Physical Qubits",
    y_axis_col="Physical Error Rate",
    value_col_to_plot="NISQ Time to Solution (hr)",
    title="Time to Estimate $\\langle Z_{tot}^2 \\rangle$ in an $11 \\times 11$ Ising Model (NISQ Mode of Operation)",
    invert_yaxis=True,
    figsize=(7.5, 2.5),
    vmin=1,
    vmax=1000,
    cbar_label="NISQ time to solution (hrs)",
)

ax.set_ylabel("Physical error rate")
ax.set_xlabel("Number of physical qubits")

plt.savefig("nisq_ising_heatmap.pdf", bbox_inches="tight")
plt.tight_layout()
plt.show()


# --- Plot FT/NISQ Ratio Heatmap ---

# Calculate the ratio
df_optimal_time_with_nisq["Ratio (FT/NISQ)"] = (
    df_optimal_time_with_nisq["Time to Solution (hr)"]
    / df_optimal_time_with_nisq["NISQ Time to Solution (hr)"]
)

# Calculate the base-10 log of the ratio for a linear color scale.
df_optimal_time_with_nisq["Log10 Ratio (FT/NISQ)"] = np.log10(
    df_optimal_time_with_nisq["Ratio (FT/NISQ)"]
)

# --- Create a Custom Asymmetric Colormap ---
# Get the data range to dynamically calculate the center position.
vmin = df_optimal_time_with_nisq["Log10 Ratio (FT/NISQ)"].min()
vmax = df_optimal_time_with_nisq["Log10 Ratio (FT/NISQ)"].max()
vcenter = 0.0
steep_range = 2

# --- Create a Custom Asymmetric Colormap ---
# Sample the standard 'RdBu_r' colormap to get its exact blue, white, and red.
rdbu_r = cm.get_cmap("RdBu_r")
blue_color = rdbu_r(0.0)
blue_white_color = rdbu_r(0.2)
white_color = rdbu_r(0.5)
white_red_color = rdbu_r(0.8)
red_color = rdbu_r(1.0)


# The normalized position of our data center (0.0) in the data's range.
if vmax == vmin or (vmax - vmin) < 2 * steep_range:
    asymmetric_cmap = "RdBu_r"
else:
    inflection_1 = (vcenter - vmin - steep_range) / (vmax - vmin)
    center_norm_pos = (vcenter - vmin) / (vmax - vmin)
    inflecttion_2 = (vcenter - vmin + steep_range) / (vmax - vmin)

    # Create the custom asymmetric colormap where 'white' is at the 5/6th position.
    asymmetric_cmap = LinearSegmentedColormap.from_list(
        "custom_asymmetric_rdbu",
        [
            (0.0, blue_color),
            (inflection_1, blue_white_color),
            (center_norm_pos, white_color),
            (inflecttion_2, white_red_color),
            (1.0, red_color),
        ],
    )


# Plot the ratio heatmap
fig, ax = flasq_plt.plot_flasq_heatmap(
    df_optimal_time_with_nisq,
    x_axis_col="Number of Physical Qubits",
    y_axis_col="Physical Error Rate",
    value_col_to_plot="Log10 Ratio (FT/NISQ)",
    title="Log$_{10}$ Runtime Ratio (Fault-Tolerant / NISQ) for Ising Simulation",
    figsize=(7.5, 5),
    invert_yaxis=True,
    log_scale=False,  # We are plotting the log of the ratio, so the scale is linear.
    cmap=asymmetric_cmap,
    cbar_label="Log$_{10}$ (FT time / NISQ time)",
    vmin=vmin,
    vmax=vmax,
    # The `center` argument is omitted; the centering is now encoded in the colormap.
)

ax.set_ylabel("Physical error rate")
ax.set_xlabel("Number of physical qubits")

plt.savefig("nisq_conservative_ratio_ising_heatmap.pdf", bbox_inches="tight")
plt.tight_layout()
plt.show()

# Comparison with Beverland et al. (2022) (4th Order)

This section performs a resource estimation to compare the FLASQ model against the results presented in Beverland et al., "The cost of quantum error correction for fault-tolerant quantum computing" (arXiv:2211.07629v1).

We focus on the "quantum dynamics" application detailed in their work: a 10x10 spin Ising model simulation using a 4th-order Trotter-Suzuki decomposition for 20 steps. The comparison is made against their "nanosecond" qubit profiles.

### FLASQ Configuration for Comparison

- **Cycle Time ($t_{cyc}$):** 400 ns. This is derived from the nanosecond profile in Beverland et al. (4 layers of 2Q gates @ 50ns, 2 layers of 1Q gates @ 50ns, and 1 layer of Measurement/Reset @ 100ns).
- **Reaction Time:** 0 cycles. This is chosen to align with the Parallel Synthesis Sequential Pauli Computation (PSSPC) model, where reaction time is not expected to be the dominant cost factor.
- **Error Budget:** A total allowable rotation synthesis error of 0.001 is used.
- **Error Mitigation:** Probabilistic Error Cancellation (PEC) is used to handle residual logical and cultivation errors, and the "Effective Time per Sample (s)" metric, which includes PEC overhead, is used for comparison.

In [ ]:
# --- Configuration for Beverland et al. Comparison ---

# Hardware and Error Model Parameters
TIME_PER_SURFACE_CODE_CYCLE_BEV = 400e-9  # 400 ns
REACTION_TIME_CYCLES_BEV = 10
TOTAL_ALLOWABLE_ROTATION_ERROR_BEV = 0.001

# Circuit Parameters
if IS_FAST_MODE:
    ROWS_BEV = 5
    COLS_BEV = 5
    N_STEPS_BEV = 5
else:
    ROWS_BEV = 10
    COLS_BEV = 10
    N_STEPS_BEV = 20

circuit_builder_kwargs_list_bev = [
    frozendict(
        {
            "rows": ROWS_BEV,
            "cols": COLS_BEV,
            "j_coupling": 1.0,
            "h_field": 0.5,  # A standard value for this comparison
            "dt": 0.1,
            "n_steps": N_STEPS_BEV,
            "order": 4,  # Use 4th-order Trotter
        }
    )
]

# Sweep Ranges
P_PHYS_BEV = [1e-4, 1e-3]
if IS_FAST_MODE:
    N_PHYS_QUBITS_BEV = [50_000, 10_000_000]
    CODE_DISTANCE_BEV = [5, 21]
else:
    N_PHYS_QUBITS_BEV = [
        int(n) for n in np.logspace(np.log10(50_000), np.log10(10_000_000), 20)
    ]
    CODE_DISTANCE_BEV = list(range(5, 41, 2))

In [ ]:
# --- Execute the Sweep for Comparison ---

# Generate CoreParametersConfig list from cultivation data
if IS_FAST_MODE:
    core_configs_bev = [
        CoreParametersConfig(
            code_distance=5,
            phys_error_rate=1e-4,
            cultivation_error_rate=1e-6,
            vcult_factor=3.0,
            cultivation_data_source_distance=5,
            target_t_error=1e-6,
        ),
        CoreParametersConfig(
            code_distance=21,
            phys_error_rate=1e-3,
            cultivation_error_rate=1e-4,
            vcult_factor=3.5,
            cultivation_data_source_distance=5,
            target_t_error=1e-4,
        ),
    ]
else:
    core_configs_bev = generate_configs_from_cultivation_data(
        code_distance_list=CODE_DISTANCE_BEV,
        phys_error_rate_list=P_PHYS_BEV,
        cultivation_data_source_distance_list=[3, 5],
        cultivation_data_sampling_frequency=1,  # Sample a few points per config
    )

# Define FLASQ models to use
flasq_model_configs_bev = [
    (conservative_FLASQ_costs, "Conservative"),
]

# Execute the sweep
sweep_results_bev = run_sweep(
    circuit_builder_func=build_ising_circuit,
    circuit_builder_kwargs_list=circuit_builder_kwargs_list_bev,
    core_configs_list=core_configs_bev,
    total_allowable_rotation_error_list=[TOTAL_ALLOWABLE_ROTATION_ERROR_BEV],
    n_phys_qubits_total_list=N_PHYS_QUBITS_BEV,
    flasq_model_configs=flasq_model_configs_bev,
    reaction_time_in_cycles_list=[REACTION_TIME_CYCLES_BEV],
    print_level=1,
)

In [ ]:
# --- Post-Process and Filter for Pareto Frontier ---

# Process the raw sweep results to get final runtimes
compare_scan_df = post_process_for_pec_runtime(
    sweep_results_bev, time_per_surface_code_cycle=TIME_PER_SURFACE_CODE_CYCLE_BEV
)

# Filter out invalid results
df_filtered_bev = compare_scan_df[
    (compare_scan_df["Effective Time per Sample (s)"] > 0)
    & (compare_scan_df["Effective Time per Sample (s)"] < 500)
].copy()

# Ensure we only look at the Conservative model (safeguard)
df_filtered_bev = df_filtered_bev[
    df_filtered_bev["FLASQ Model"] == "Conservative"
].copy()

# Find the Pareto frontier: the optimal configuration (min time) for each
# combination of (Qubits, Error Rate).
min_time_indices_bev = df_filtered_bev.groupby(
    ["Number of Physical Qubits", "Physical Error Rate"]
)["Effective Time per Sample (s)"].idxmin()

df_optimal_bev = (
    df_filtered_bev.loc[min_time_indices_bev]
    .reset_index(drop=True)
    .sort_values("Number of Physical Qubits")
)


# Define a helper function for formatting error rates to match the desired style (0.0001, 0.001)
# Use np.isclose for robust float comparison.
def format_error_rate(p):
    if np.isclose(p, 1e-4):
        return "0.0001"
    elif np.isclose(p, 1e-3):
        return "0.001"
    return f"{p}"  # Fallback


# Apply formatting for the legend
df_optimal_bev["p_phys_label"] = df_optimal_bev["Physical Error Rate"].apply(
    format_error_rate
)

In [ ]:
# --- Visualization: FLASQ vs. Beverland et al. ---
from matplotlib.lines import Line2D  # Import Line2D for manual legend handles

# Define Beverland et al. target data points (From Table IV of their paper)
beverland_data = [
    {
        "p_phys": 1e-3,
        "Qubits": 8.2e6,
        "Runtime": 1.1,
        "Type": "Time-opt ($p_{phys} =$ 1e-3)",
    },
    {
        "p_phys": 1e-3,
        "Qubits": 0.94e6,
        "Runtime": 13,
        "Type": "Space-opt ($p_{phys} =$ 1e-3)",
    },
    {
        "p_phys": 1e-4,
        "Qubits": 0.68e6,
        "Runtime": 0.56,
        "Type": "Time-opt ($p_{phys} =$ 1e-4)",
    },
    {
        "p_phys": 1e-4,
        "Qubits": 0.11e6,
        "Runtime": 6.7,
        "Type": "Space-opt ($p_{phys} =$ 1e-4)",
    },
]
df_bev_points = pd.DataFrame(beverland_data)
# Apply the same formatting for consistency
df_bev_points["p_phys_label"] = df_bev_points["p_phys"].apply(format_error_rate)

# --- Plotting Setup ---
sns.set_theme(style="whitegrid")
# Use plt.subplots to get fig and ax handles explicitly
fig, ax = plt.subplots(figsize=(7.5, 5))

# Define Palettes and Styles
palette = sns.color_palette("colorblind")

# Define labels and map them to colors
P_1E4_LABEL = format_error_rate(1e-4)
P_1E3_LABEL = format_error_rate(1e-3)
error_rate_palette = {
    P_1E4_LABEL: palette[0],  # Blue
    P_1E3_LABEL: palette[1],  # Orange
}

# Define specific markers for the Beverland points
marker_styles = {
    "Time-opt ($p_{phys} =$ 1e-3)": "o",
    "Space-opt ($p_{phys} =$ 1e-3)": "X",
    "Time-opt ($p_{phys} =$ 1e-4)": "s",
    "Space-opt ($p_{phys} =$ 1e-4)": "P",  # 'P' is a filled plus sign
}

# Plot FLASQ Pareto frontiers (Lines)
# Set legend=False to handle it manually. Remove 'style' as it's now constant.
sns.lineplot(
    data=df_optimal_bev,
    x="Number of Physical Qubits",
    y="Effective Time per Sample (s)",
    hue="p_phys_label",
    # style="FLASQ Model", # Removed
    palette=error_rate_palette,
    lw=2.5,
    alpha=0.8,
    ax=ax,
    legend=False,  # Disable automatic legend
)

# Plot Beverland et al. target points (Markers)
# Set legend=False to handle it manually
sns.scatterplot(
    data=df_bev_points,
    x="Qubits",
    y="Runtime",
    hue="p_phys_label",
    style="Type",
    palette=error_rate_palette,
    markers=marker_styles,
    s=200,
    ax=ax,
    zorder=5,
    legend=False,  # Disable automatic legend
)

# Configure the plot (Title, Labels, Scale)
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_title(
    "Ising Model Runtime Comparison: FLASQ (with PEC) vs Beverland et al. (2022)",
    # fontsize=16,
)
ax.set_xlabel("Number of physical qubits")
ax.set_ylabel("Effective runtime per sample (s)")
ax.grid(True, which="both", ls="--", alpha=0.6)

# --- Manual Legend Construction ---

# 1. Create handles for the lines (FLASQ Curves)
line_handles = [
    Line2D([0], [0], color=error_rate_palette[P_1E3_LABEL], lw=2.5),
    Line2D([0], [0], color=error_rate_palette[P_1E4_LABEL], lw=2.5),
]
# line_labels = [P_1E4_LABEL, P_1E3_LABEL]
line_labels = ["$p_{phys} =$ 1e-3", "$p_{phys} =$ 1e-4"]
# line_labels = ["$p_{phys} = 10^{-4}$", "$p_{phys} = 10^{-3}$"]

# 2. Create handles for the markers (Beverland Points)
marker_handles = []
marker_labels = []
# Define the order for the legend display (matching the target image)
bev_types_ordered = [
    "Time-opt ($p_{phys} =$ 1e-3)",
    "Space-opt ($p_{phys} =$ 1e-3)",
    "Time-opt ($p_{phys} =$ 1e-4)",
    "Space-opt ($p_{phys} =$ 1e-4)",
]

for bev_type in bev_types_ordered:
    # Determine the color based on the error rate in the label
    if "1e-4" in bev_type:
        color = error_rate_palette[P_1E4_LABEL]
    else:
        color = error_rate_palette[P_1E3_LABEL]

    # Create the handle (use color='w' for the line part of the marker handle)
    # Use a slight edge color ('k'=black) for better visibility, matching the target image style
    handle = Line2D(
        [0],
        [0],
        marker=marker_styles[bev_type],
        color="w",
        markerfacecolor=color,
        markersize=10,
        markeredgecolor="k",
        markeredgewidth=0.5,
        lw=0,
    )
    marker_handles.append(handle)
    marker_labels.append(bev_type)

# 3. Assemble the final legend
# Use blank handles for titles/separators
blank_handle = Line2D([0], [0], color="w", alpha=0)


# Combine handles and labels in the desired structure
legend_handles = (
    [
        blank_handle,  # Title for Physical Error Rate
    ]
    + line_handles
    + [
        blank_handle,  # Title for Type
    ]
    + marker_handles
)

legend_labels = (
    [
        "FLASQ estimates",
    ]
    + line_labels
    + [
        "Beverland et al. (2022)",
    ]
    + marker_labels
)

# Create the legend
# ax.legend(handles=legend_handles, labels=legend_labels, loc="upper right")
ax.legend(
    handles=legend_handles,
    labels=legend_labels,
    bbox_to_anchor=(1.0, 0.0),
    loc="lower right",
    labelspacing=0.45,
    framealpha=0.9,
)

ax.set_ylim(bottom=0.02)

plt.tight_layout()
plt.savefig("flasq_vs_beverland_loglog_conservative.pdf", bbox_inches="tight")
plt.show()

# Comparison with Beverland et al. (2022) - High-Fidelity (Pure QEC)

This section performs a second comparison against Beverland et al. (2022), but this time matching their error methodology. Instead of using Probabilistic Error Cancellation (PEC) to handle residual errors, we demand that Quantum Error Correction (QEC) alone suppresses the failure probability below a fixed budget.

### Methodology

We adopt the total error budget $\epsilon_{total} = 0.001$ used in the target paper and partition it equally:
- $\epsilon_{syn}$ (Synthesis Bias) = 0.001 / 3
- $\epsilon_{dis}$ (Cultivation/T-gate Failure) = 0.001 / 3
- $\epsilon_{log}$ (Clifford/Memory Failure) = 0.001 / 3

We use the `generate_configs_for_constrained_qec` utility, which optimizes the cultivation strategy *before* the sweep by calculating the required magic state fidelity ($p_{mag}$) based on the T-count and $\epsilon_{dis}$.

The sweep is then executed, and `post_process_for_failure_budget` filters the results to ensure the resulting Clifford failures ($E_{fail, Clifford}$) are below $\epsilon_{log}$.

The final metric for comparison is the raw "Wall Clock Time (s)", as no sampling overhead is incurred.

In [ ]:
# --- Configuration for High-Fidelity Comparison ---

# Define the Error Budget (Total = 0.001, split equally)
ERROR_BUDGET_TOTAL = 0.001
error_budget_high_fidelity = ErrorBudget(
    logical=ERROR_BUDGET_TOTAL / 3,
    cultivation=ERROR_BUDGET_TOTAL / 3,
    synthesis=ERROR_BUDGET_TOTAL / 3,
)

# We reuse the hardware and circuit parameters defined in the previous comparison section:
# TIME_PER_SURFACE_CODE_CYCLE_BEV (400ns)
# P_PHYS_BEV ([1e-4, 1e-3])
# N_PHYS_QUBITS_BEV (50k to 10M)
# CODE_DISTANCE_BEV (5 to 41)
# circuit_builder_kwargs_list_bev (10x10, 4th order, 20 steps)

# We also reuse the model config (Conservative)
# flasq_model_configs_bev = [(conservative_FLASQ_costs, "Conservative")]

In [ ]:
# --- Generate Optimized Configurations (Constrained QEC) ---

# Generate the configurations by optimizing cultivation based on the error budget.
# This replaces the standard generate_configs_from_cultivation_data call.

# We assume only one circuit configuration is being analyzed here.
circuit_kwargs_bev = circuit_builder_kwargs_list_bev[0]

print("Generating optimized configurations based on error budget...")
core_configs_qec = generate_configs_for_constrained_qec(
    circuit_builder_func=build_ising_circuit,
    circuit_builder_kwargs=circuit_kwargs_bev,
    error_budget=error_budget_high_fidelity,
    phys_error_rate_list=P_PHYS_BEV,
    code_distance_list=CODE_DISTANCE_BEV,
    # Using default parameters for cultivation data analysis
)

print(f"Generated {len(core_configs_qec)} optimized configurations.")

In [ ]:
# --- Execute the Sweep (High Fidelity) ---

if not core_configs_qec:
    print("No feasible configurations generated. Skipping sweep.")
    sweep_results_qec = []
else:
    print("Executing sweep with optimized configurations...")
    sweep_results_qec = run_sweep(
        circuit_builder_func=build_ising_circuit,
        circuit_builder_kwargs_list=[circuit_kwargs_bev],
        core_configs_list=core_configs_qec,
        # Crucial: Use the synthesis budget for the rotation error parameter
        total_allowable_rotation_error_list=[error_budget_high_fidelity.synthesis],
        n_phys_qubits_total_list=N_PHYS_QUBITS_BEV,
        flasq_model_configs=flasq_model_configs_bev,  # Using Conservative model
        reaction_time_in_cycles_list=[REACTION_TIME_CYCLES_BEV],
        print_level=1,
    )

In [ ]:
# --- Post-Process and Filter for Pareto Frontier ---

# Process the results using the failure budget post-processor
# This filters results that fail to meet the logical/cultivation budgets.
df_qec = post_process_for_failure_budget(
    sweep_results_qec,
    error_budget=error_budget_high_fidelity,
    time_per_surface_code_cycle=TIME_PER_SURFACE_CODE_CYCLE_BEV,
)

if df_qec.empty:
    print("No configurations found that satisfy the error budget.")
    df_optimal_qec = pd.DataFrame()
else:
    print(f"Found {len(df_qec)} configurations satisfying the budget.")

    # Ensure we only look at the Conservative model (safeguard, if multiple models were run)
    df_filtered_qec = df_qec[df_qec["FLASQ Model"] == "Conservative"].copy()

    # Find the Pareto frontier: the optimal configuration (min time) for each
    # combination of (Qubits, Error Rate).
    # The metric to optimize is now "Wall Clock Time (s)".
    min_time_indices_qec = df_filtered_qec.groupby(
        ["Number of Physical Qubits", "Physical Error Rate"]
    )["Wall Clock Time (s)"].idxmin()

    df_optimal_qec = (
        df_filtered_qec.loc[min_time_indices_qec]
        .reset_index(drop=True)
        .sort_values("Number of Physical Qubits")
    )

    # Apply formatting for the legend (reusing the helper from the previous section)
    df_optimal_qec["p_phys_label"] = df_optimal_qec["Physical Error Rate"].apply(
        format_error_rate
    )

In [ ]:
# --- Visualization: FLASQ (High-Fidelity QEC) vs. Beverland et al. ---

if df_optimal_qec.empty:
    print("Skipping visualization as no valid configurations were found.")
else:
    # --- Plotting Setup ---
    sns.set_theme(style="whitegrid")
    fig, ax = plt.subplots(figsize=(7.5, 5))

    # We reuse the palettes and styles defined in the previous plotting cell
    # error_rate_palette, marker_styles, P_1E4_LABEL, P_1E3_LABEL

    # Plot FLASQ High-Fidelity Pareto frontiers (Lines)
    sns.lineplot(
        data=df_optimal_qec,
        x="Number of Physical Qubits",
        y="Wall Clock Time (s)",  # NOTE: Changed Y-axis metric
        hue="p_phys_label",
        palette=error_rate_palette,
        lw=2.5,
        alpha=0.8,
        ax=ax,
        legend=False,
    )

    # Plot Beverland et al. target points (Markers)
    # df_bev_points was defined in the previous plotting cell
    sns.scatterplot(
        data=df_bev_points,
        x="Qubits",
        y="Runtime",
        hue="p_phys_label",
        style="Type",
        palette=error_rate_palette,
        markers=marker_styles,
        s=200,
        ax=ax,
        zorder=5,
        legend=False,
    )

    # Configure the plot
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_title(
        "Ising Model Runtime Comparison: FLASQ (high fidelity) vs Beverland et al. (2022)",
        # fontsize=16,
    )
    ax.set_xlabel("Number of physical qubits")
    ax.set_ylabel("Wall clock time per sample (s)")  # Updated Label
    ax.grid(True, which="both", ls="--", alpha=0.6)

    # --- Manual Legend Construction ---
    # Reusing the legend construction logic from the previous plot cell
    # (This assumes legend_handles and legend_labels were defined in the previous cell's scope)

    # If the previous plot cell was not run, we must reconstruct the handles/labels:
    from matplotlib.lines import Line2D

    # 1. Create handles for the lines (FLASQ Curves)
    line_handles = [
        # Line2D([0], [0], color=error_rate_palette[P_1E3_LABEL], lw=2.5),
        Line2D([0], [0], color=error_rate_palette[P_1E4_LABEL], lw=2.5),
    ]
    # line_labels = [P_1E4_LABEL, P_1E3_LABEL]
    # line_labels = ["$p_{phys} =$ 1e-3", "$p_{phys} =$ 1e-4"]
    line_labels = ["$p_{phys} =$ 1e-4"]

    # 2. Create handles for the markers (Beverland Points)
    marker_handles = []
    marker_labels = []
    bev_types_ordered = [
        "Time-opt ($p_{phys} =$ 1e-3)",
        "Space-opt ($p_{phys} =$ 1e-3)",
        "Time-opt ($p_{phys} =$ 1e-4)",
        "Space-opt ($p_{phys} =$ 1e-4)",
    ]

    for bev_type in bev_types_ordered:
        color = (
            error_rate_palette[P_1E4_LABEL]
            if "1e-4" in bev_type
            else error_rate_palette[P_1E3_LABEL]
        )
        handle = Line2D(
            [0],
            [0],
            marker=marker_styles[bev_type],
            color="w",
            markerfacecolor=color,
            markersize=10,
            markeredgecolor="k",
            markeredgewidth=0.5,
            lw=0,
        )
        marker_handles.append(handle)
        marker_labels.append(bev_type)

    # 3. Assemble the final legend
    blank_handle = Line2D([0], [0], color="w", alpha=0)
    legend_handles = [blank_handle] + line_handles + [blank_handle] + marker_handles
    legend_labels = (
        ["FLASQ estimates"] + line_labels + ["Beverland et al. (2022)"] + marker_labels
    )
    # End of legend reconstruction block

    # Create the legend
    ax.legend(
        handles=legend_handles,
        labels=legend_labels,
        bbox_to_anchor=(1.0, 0.0),
        loc="lower right",
        labelspacing=0.45,
        framealpha=0.9,
    )

    ax.set_ylim(bottom=0.02)

    plt.tight_layout()
    plt.savefig("flasq_vs_beverland_loglog_high_fidelity.pdf", bbox_inches="tight")
    plt.show()

### T-Count for the Beverland et al. Comparison

To provide additional context for the comparisons, we can explicitly calculate the total number of T-gates required for the 4th-order Ising model simulation, given the rotation synthesis error budget of $\epsilon_{syn} = 0.001 / 3$.

We follow the same procedure as the single-point analysis at the top of the notebook, using the `apply_flasq_cost_model` infrastructure to ensure a consistent calculation.

In [ ]:
# --- Calculate T-Count for the Beverland et al. circuit ---

# 1. Get the circuit parameters and error budget from the comparison section
circuit_kwargs_bev = circuit_builder_kwargs_list_bev[0]
rotation_synthesis_error_bev = 0.001

# 2. Build and analyze the logical circuit to get symbolic counts.
circuit_bev = build_ising_circuit(**circuit_kwargs_bev)
cbloq_bev, _ = convert_circuit_for_flasq_analysis(circuit_bev)

flasq_counts_bev = get_cost_value(cbloq_bev, FLASQGateTotals())
total_span_bev = get_cost_value(cbloq_bev, TotalSpanCost())
qubit_counts_bev = get_cost_value(cbloq_bev, QubitCount())

ind_rot_err_bev = rotation_synthesis_error_bev / flasq_counts_bev.total_rotations
rotation_depth_val_bev = get_rotation_depth(rotation_error=ind_rot_err_bev)
measurement_depth_bev = get_cost_value(
    cbloq_bev, TotalMeasurementDepth(rotation_depth=rotation_depth_val_bev)
)

# 3. Calculate T-count directly without calling the full cost model
#    to avoid n_fluid_ancilla = 0 producing NaN in fields we don't need.
total_t_count_bev = (
    flasq_counts_bev.t
    + flasq_counts_bev.toffoli * 4
    + flasq_counts_bev.and_gate * 4
    + MIXED_FALLBACK_T_COUNT * (flasq_counts_bev.z_rotation + flasq_counts_bev.x_rotation)
)

# 4. Resolve symbols to get the final numeric T-count.
resolved_t_count_bev = substitute_until_fixed_point(
    total_t_count_bev,
    frozendict({ROTATION_ERROR: ind_rot_err_bev}),
    try_make_number=True
)

resolved_rotation_count_bev = flasq_counts_bev.x_rotation + flasq_counts_bev.z_rotation

print("--- T-Count Summary for Beverland et al. Comparison ---")
print(
    f"Circuit: {circuit_kwargs_bev['rows']}x{circuit_kwargs_bev['cols']} Ising Model, {circuit_kwargs_bev['order']}th order, {circuit_kwargs_bev['n_steps']} steps"
)
print(f"Rotation Synthesis Error Budget (ε_syn): {rotation_synthesis_error_bev:.2e}")
print("-" * 55)
print(f"Total Number of Rotations: {int(resolved_rotation_count_bev):,}")
print(f"Total T-Gate Count: {int(resolved_t_count_bev):,}")
print("-" * 55)

### Detailed Analysis of a Single Beverland Comparison Point

This cell performs a detailed breakdown of a single, interesting data point from the `df_optimal_bev` table: the one corresponding to the minimum number of physical qubits for a `p_phys` of 0.001. This allows us to inspect the underlying resource counts and verify that our calculations align with the summary table.

In [ ]:
# 1. Isolate the Target Data Point
# We select the row with a Physical Error Rate of 0.001 and the minimum number of qubits.

# 2. Extract Key Parameters from the Target Row
physical_error_rate_bev = 0.001
code_distance_bev = 14
n_fluid_ancilla_bev = 40
# code_distance_bev = target_row["Code Distance"]
# n_fluid_ancilla_bev = target_row["Number of Fluid Ancilla Qubits"]

# Set a target logical error rate for cultivation. You can tweak this value.
# (This is set so that the assert below passes - a little janky but it'll do for now.)
target_logical_error_rate_bev = 2e-7

# Find the best cultivation parameters for the given target.
best_cult_params_bev = cultivation_analysis.find_best_cultivation_parameters(
    physical_error_rate=physical_error_rate_bev,
    target_logical_error_rate=target_logical_error_rate_bev,
)

# Derive the V_CULT factor from the cultivation data.
expected_volume_bev = best_cult_params_bev["expected_volume"]
vcult_factor_derived_bev = expected_volume_bev / (
    2 * (code_distance_bev + 1) ** 2 * code_distance_bev
)
cultivation_error_rate_bev = best_cult_params_bev["t_gate_cultivation_error_rate"]


# Assert that our derived factor matches the one from the optimal sweep.

# The number of algorithmic qubits is fixed by the circuit arguments
n_algorithmic_qubits_bev = 100
n_total_logical_qubits_bev = n_algorithmic_qubits_bev + n_fluid_ancilla_bev

# 3. Build and Analyze the Logical Circuit (reusing from previous section)
# The circuit is the same for all points in the Beverland comparison.
circuit_bev = build_ising_circuit(**circuit_builder_kwargs_list_bev[0])
cbloq_bev, _ = convert_circuit_for_flasq_analysis(circuit_bev)

flasq_counts_bev = get_cost_value(cbloq_bev, FLASQGateTotals())
total_span_bev = get_cost_value(cbloq_bev, TotalSpanCost())
qubit_counts_bev = get_cost_value(cbloq_bev, QubitCount())

ind_rot_err_bev = TOTAL_ALLOWABLE_ROTATION_ERROR_BEV / flasq_counts_bev.total_rotations
rotation_depth_val_bev = get_rotation_depth(rotation_error=ind_rot_err_bev)
measurement_depth_bev = get_cost_value(
    cbloq_bev, TotalMeasurementDepth(rotation_depth=rotation_depth_val_bev)
)

# 4. Apply the FLASQ Cost Model
flasq_summary_bev = apply_flasq_cost_model(
    model=conservative_FLASQ_costs,
    n_total_logical_qubits=n_total_logical_qubits_bev,
    qubit_counts=qubit_counts_bev,
    counts=flasq_counts_bev,
    span_info=total_span_bev,
    measurement_depth=measurement_depth_bev,
    logical_timesteps_per_measurement=REACTION_TIME_CYCLES_BEV / code_distance_bev,
)

# 5. Resolve Symbols and Calculate Final Metrics
resolved_flasq_summary_bev = flasq_summary_bev.resolve_symbols(
    frozendict(
        {
            V_CULT_FACTOR: vcult_factor_derived_bev,
            ROTATION_ERROR: ind_rot_err_bev,

            T_REACT: REACTION_TIME_CYCLES_BEV / code_distance_bev,
        }
    )
)

eff_time_bev, wall_time_bev, gamma_per_block = calculate_error_mitigation_metrics(
    flasq_summary=resolved_flasq_summary_bev,
    time_per_surface_code_cycle=TIME_PER_SURFACE_CODE_CYCLE_BEV,
    code_distance=code_distance_bev,
    lambda_val=0.01 / physical_error_rate_bev,
    cultivation_error_rate=cultivation_error_rate_bev,
)

P_fail_Clifford, P_fail_T = calculate_failure_probabilities(
    flasq_summary=resolved_flasq_summary_bev,
    code_distance=code_distance_bev,
    lambda_val=0.01 / physical_error_rate_bev,
    cultivation_error_rate=cultivation_error_rate_bev,
)

# Calculate non-Clifford cost contributions for detailed reporting
total_non_clifford_volume_bev = (
    resolved_flasq_summary_bev.non_clifford_lattice_surgery_volume
    + resolved_flasq_summary_bev.cultivation_volume
)
frac_comp_vol_non_clifford_bev = (
    total_non_clifford_volume_bev
    / resolved_flasq_summary_bev.total_computational_volume
)
frac_non_clifford_vol_cultivation_bev = (
    resolved_flasq_summary_bev.cultivation_volume / total_non_clifford_volume_bev
)

frac_spacetime_vol_non_clifford_bev = (
    total_non_clifford_volume_bev / resolved_flasq_summary_bev.total_spacetime_volume
)

# 6. Print a Clean Summary of the Results
print("--- Detailed Analysis for Beverland Comparison Point ---")
print(f"  Physical Error Rate: {physical_error_rate_bev:.1e}")
print(f"  Code Distance: {code_distance_bev}")
print(f"  Target Logical Error Rate (Cultivation): {target_logical_error_rate_bev:.1e}")
print("-" * 50)
print("  Derived Cultivation Parameters:")
print(f"    Derived V_CULT Factor: {vcult_factor_derived_bev:.4f} (matches row value)")
print(f"    Cultivation Error Rate: {cultivation_error_rate_bev:.10f}")
print(
    f"  Total Physical Qubits: {n_total_logical_qubits_bev * 2 * (code_distance_bev + 1)**2}"
)
print("-" * 50)
print("  Resource Summary:")
print(f"    Algorithmic Qubits: {resolved_flasq_summary_bev.n_algorithmic_qubits}")
print(f"    Fluid Ancilla Qubits: {resolved_flasq_summary_bev.n_fluid_ancilla}")
print(f"    Total Rotations: {resolved_flasq_summary_bev.total_rotation_count:,.0f}")
print(f"    Total T-gates: {resolved_flasq_summary_bev.total_t_count:,.0f}")
print(f"    Total Logical Depth: {resolved_flasq_summary_bev.total_depth:,.2f}")
print("-" * 50)
print("  Volume Breakdown:")
print(
    f"    Total Computational Volume: {resolved_flasq_summary_bev.total_computational_volume:,.2f}"
)
print(
    f"    Total Spacetime Volume: {resolved_flasq_summary_bev.total_spacetime_volume:,.2f}"
)
print(f"    Total Non-Clifford Volume: {total_non_clifford_volume_bev:,.2f}")
print(f"      Fraction of Computational Volume: {frac_comp_vol_non_clifford_bev:.2%}")
print(f"      Fraction of Spacetime Volume: {frac_spacetime_vol_non_clifford_bev:.2%}")
print(
    f"      Fraction of Non-Clifford from Cultivation: {frac_non_clifford_vol_cultivation_bev:.2%}"
)
print("-" * 50)
print("  Final Performance Metrics (re-calculated):")
print(f"    Effective Time per Sample: {eff_time_bev:.4f} s")
print(f"    Wall Clock Time per Sample: {wall_time_bev:.4f} s")
print("-" * 50)
print("  Error Budget Breakdown:")
print(f"    Probability of Clifford Failure (P_fail_Clifford): {P_fail_Clifford:.4f}")
print(f"    Probability of T-gate Failure (P_fail_T): {P_fail_T:.4f}")
print(
    f"    Cumulative Failure Probability: {(1 - (1 - P_fail_Clifford) * (1 - P_fail_T)):.4f}"
)
# print("-" * 50) # Removed as this is now handled by the unified function

### FLASQ Estimates with Open Boundary Conditions and more Trotter steps

We now repeat the analysis but for a system with open boundary conditions (OBC). This is more representative of what might be implemented on a near-term device without costly long-range interactions. The resulting heatmap will serve as the basis for a more direct comparison with the NISQ cost model.

In [ ]:
# 3. Post-process the OBC results.
ising_scan_df_obc_deep = post_process_for_pec_runtime(
    sweep_results_obc_deep, time_per_surface_code_cycle=time_per_surface_code_cycle
)
df_enriched_obc_deep = flasq_plt.enrich_sweep_df(
    ising_scan_df_obc_deep, target_std_dev=target_std_deviation
)

In [ ]:
# --- Plot the OBC Time to Solution Heatmap ---

# 1. Filter the enriched obc_deep data for the 'Conservative' model.
df_conservative_obc_deep = df_enriched_obc_deep[
    df_enriched_obc_deep["FLASQ Model"] == "Conservative"
]

print(df_conservative_obc_deep)

df_optimal_time_obc_deep_filtered = df_conservative_obc_deep[
    (df_conservative_obc_deep["Effective Time per Sample (s)"] > 0)
    & (df_conservative_obc_deep["Time to Solution (hr)"] > 0)
    & (df_conservative_obc_deep["Time to Solution (hr)"] <= 1000)
    & (df_conservative_obc_deep["Physical Error Rate"] >= 4e-4)
    & (df_conservative_obc_deep["Number of Physical Qubits"] <= 400000)
]

# Find the optimal FT configurations from the OBC data.
df_optimal_ft_obc_deep = flasq_plt.find_optimal_heatmap_configs(
    df_optimal_time_obc_deep_filtered,
    x_axis_col="Number of Physical Qubits",
    y_axis_col="Physical Error Rate",
    value_col_to_optimize="Time to Solution (hr)",
    regularization_col=None,
)

# 3. Plot the heatmap for OBC Deep (Added to match paper audit).
fig, ax = flasq_plt.plot_flasq_heatmap(
    df_optimal_ft_obc_deep,
    x_axis_col="Number of Physical Qubits",
    y_axis_col="Physical Error Rate",
    value_col_to_plot="Time to Solution (hr)",
    title="Time to Estimate $\\langle Z_{tot}^2 \\rangle$ (Open Boundary Conditions, 40 Trotter Steps)",
    invert_yaxis=True,
    vmin=1,
    vmax=1000,
    cbar_label="Time to solution (hrs)",
)
ax.set_ylabel("Physical error rate")
ax.set_xlabel("Number of physical qubits")
plt.savefig("conservative_ising_heatmap_obc_deep.pdf", bbox_inches="tight")
plt.tight_layout()
plt.show()

df_optimal_time_obc_deep_filtered = df_conservative_obc_deep[
    (df_conservative_obc_deep["Effective Time per Sample (s)"] > 0)
    & (df_conservative_obc_deep["Time to Solution (hr)"] > 0)
    & (df_conservative_obc_deep["Time to Solution (hr)"] <= 1e30)
    & (df_conservative_obc_deep["Physical Error Rate"] >= 4e-4)
    & (df_conservative_obc_deep["Number of Physical Qubits"] <= 400000)
]

# Find the optimal FT configurations from the OBC data.
df_optimal_ft_obc_deep = flasq_plt.find_optimal_heatmap_configs(
    df_optimal_time_obc_deep_filtered,
    x_axis_col="Number of Physical Qubits",
    y_axis_col="Physical Error Rate",
    value_col_to_optimize="Time to Solution (hr)",
    regularization_col=None,
)


df_optimal_time_with_nisq = add_nisq_costs_to_dataframe(
    df_optimal_ft_obc_deep.copy(),
    system_size=rows * cols,
    n_trotter_steps=40,
    target_std_dev=target_std_deviation,
    two_qubit_errors_only=True,
)

df_optimal_time_with_nisq = df_optimal_time_with_nisq[
    (df_optimal_time_with_nisq["NISQ Time to Solution (hr)"] > 0)
    & (df_optimal_time_with_nisq["NISQ Time to Solution (hr)"] < 1e30)
    & (
        (df_optimal_time_with_nisq["Time to Solution (hr)"] < 1000)
        | (df_optimal_time_with_nisq["NISQ Time to Solution (hr)"] < 1000)
    )
    & (df_optimal_time_with_nisq["Number of Physical Qubits"] >= 22000)
]

df_optimal_time_with_nisq_only = df_optimal_time_with_nisq[
    (df_optimal_time_with_nisq["NISQ Time to Solution (hr)"] < 1e3)
]

print(df_optimal_time_with_nisq_only)

# --- Plot NISQ-only Heatmap ---
fig, ax = flasq_plt.plot_flasq_heatmap(
    df_optimal_time_with_nisq_only,
    x_axis_col="Number of Physical Qubits",
    y_axis_col="Physical Error Rate",
    value_col_to_plot="NISQ Time to Solution (hr)",
    title="Time to Estimate $\\langle Z_{tot}^2 \\rangle$ in an $11 \\times 11$ Ising Model (NISQ Mode of Operation, 40 Trotter Steps)",
    invert_yaxis=True,
    figsize=(7.5, 2.5),
    vmin=1,
    vmax=1000,
    cbar_label="NISQ time to solution (hrs)",
)

ax.set_ylabel("Physical error rate")
ax.set_xlabel("Number of physical qubits")

# plt.savefig("nisq_ising_heatmap.pdf", bbox_inches="tight")
plt.tight_layout()
plt.show()


# --- Plot FT/NISQ Ratio Heatmap ---

# Calculate the ratio
df_optimal_time_with_nisq["Ratio (FT/NISQ)"] = (
    df_optimal_time_with_nisq["Time to Solution (hr)"]
    / df_optimal_time_with_nisq["NISQ Time to Solution (hr)"]
)

# Calculate the base-10 log of the ratio for a linear color scale.
df_optimal_time_with_nisq["Log10 Ratio (FT/NISQ)"] = np.log10(
    df_optimal_time_with_nisq["Ratio (FT/NISQ)"]
)

# --- Create a Custom Asymmetric Colormap ---
# Get the data range to dynamically calculate the center position.
vmin = df_optimal_time_with_nisq["Log10 Ratio (FT/NISQ)"].min()
vmax = df_optimal_time_with_nisq["Log10 Ratio (FT/NISQ)"].max()
vcenter = 0.0
steep_range = 1

# --- Create a Custom Asymmetric Colormap ---
# Sample the standard 'RdBu_r' colormap to get its exact blue, white, and red.
rdbu_r = cm.get_cmap("RdBu_r")
blue_color = rdbu_r(0.0)
blue_white_color = rdbu_r(0.2)
white_color = rdbu_r(0.5)
white_red_color = rdbu_r(0.8)
red_color = rdbu_r(1.0)

asymmetric_cmap = LinearSegmentedColormap.from_list(
    "custom_asymmetric_rdbu",
    [
        (0.0, blue_color),
        (1.0, blue_white_color),
    ],
)



# # The normalized position of our data center (0.0) in the data's range.
# if vmax == vmin or (vcenter - vmin) < steep_range or (vmax - vcenter) < steep_range:
#     asymmetric_cmap = "RdBu_r"
# else:
#     inflection_1 = (vcenter - vmin - steep_range) / (vmax - vmin)
#     center_norm_pos = (vcenter - vmin) / (vmax - vmin)
#     inflecttion_2 = (vcenter - vmin + steep_range) / (vmax - vmin)

#     # Create the custom asymmetric colormap where 'white' is at the 5/6th position.
#     asymmetric_cmap = LinearSegmentedColormap.from_list(
#         "custom_asymmetric_rdbu",
#         [
#             (0.0, blue_color),
#             (inflection_1, blue_white_color),
#             (center_norm_pos, white_color),
#             (inflecttion_2, white_red_color),
#             (1.0, red_color),
#         ],
#     )


# Plot the ratio heatmap
fig, ax = flasq_plt.plot_flasq_heatmap(
    df_optimal_time_with_nisq,
    x_axis_col="Number of Physical Qubits",
    y_axis_col="Physical Error Rate",
    value_col_to_plot="Log10 Ratio (FT/NISQ)",
    title="Log$_{10}$ Runtime Ratio (Fault-Tolerant / NISQ) for Ising Simulation (40 Trotter Steps)",
    figsize=(7.5, 5),
    invert_yaxis=True,
    log_scale=False,  # We are plotting the log of the ratio, so the scale is linear.
    cmap=asymmetric_cmap,
    cbar_label="Log$_{10}$ (FT time / NISQ time)",
    vmin=vmin,
    vmax=vmax,
    # The `center` argument is omitted; the centering is now encoded in the colormap.
)

ax.set_ylabel("Physical error rate")
ax.set_xlabel("Number of physical qubits")

plt.savefig("nisq_conservative_ratio_ising_heatmap_40.pdf", bbox_inches="tight")
plt.tight_layout()
plt.show()